In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
"""
Arabic Sentiment Analysis - Fine-tuning AraBERT
Transformer-based Model for 3-class Sentiment Classification
"""

import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, classification_report




In [3]:
# ============================================================================
# Configuration
# ============================================================================
# DATA_PATH = ""
MODEL_NAME = 'aubmindlab/bert-base-arabertv02'
NUM_LABELS = 3
BATCH_SIZE = 2
EPOCHS = 4
LEARNING_RATE = 1e-4
OUTPUT_DIR = "/content/drive/MyDrive/arabert_checkpoints"

# Label mappings
id2label = {0: 'Positive', 1: 'Mixed', 2: 'Negative'}
label2id = {'Positive': 0, 'Mixed': 1, 'Negative': 2}

In [4]:
# ============================================================================
# Load Data
# ============================================================================
print("Loading datasets...")
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")


Loading datasets...


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/AraT5-base")
sample_df = train_df.head(1000)

texts = sample_df["text"].astype(str).tolist()

lengths = []

for t in texts:
    enc = tokenizer(
        t,
        add_special_tokens=True,
        truncation=False  
    )
    lengths.append(len(enc["input_ids"]))

lengths = np.array(lengths)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/2.44M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [6]:
print("Min:", lengths.min())
print("Mean:", lengths.mean())
print("Median:", np.median(lengths))
print("90th percentile:", np.percentile(lengths, 90))
print("95th percentile:", np.percentile(lengths, 95))
print("99th percentile:", np.percentile(lengths, 99))
print("Max:", lengths.max())


Min: 2
Mean: 69.458
Median: 37.5
90th percentile: 150.10000000000002
95th percentile: 238.0999999999999
99th percentile: 503.0999999999999
Max: 1520


In [7]:
train_df.head()

,label,text
0,Positive,اقم علي الماشي . بصورة عامة الغرفة جيدة وكل شي...
1,Negative,لم تعجبني كباقي السلسلة و غير متحمس لقراءة الج...
2,Positive,كان المكان ممتاز والامن والاستقبال اوكي . . ال...
3,Negative,القصة اجمالا مشوقة كونها تعتمد علي شخصيات يجمع...
4,Mixed,اربع نجوم لولا الملل الذي اصابني في النهاية......


In [8]:
t

'دقة الوصف ف كل حاجة قدرت بجد توصل احساس ابطال القصة للقارء. الماساة بتاعة سقوط الاندلس وتاثيرها علي اوربا كلها بعد ما كان الاسلام ف الفترة دي خلق مفهوم جديد لتعايش الناس اللي اديانهم مختلفة و تاثير الكلام ده اللي هو مزج الثقافات طلعلنا علماء كتير من الاندلس ومن البلاد المجاورة زي دافنشي ف ايطاليا .. الكاتبة وصفت جهل ووحشية و المملكة القشتالية وحكم الكنيسة ف الوقت ده والمفهوم الضيق وعدم القناعة بفكرة التعايش ف مشهد حرق الكتب واد ايه اتاثروابيه العرب اللي كانوا عايشين هناك و غيرها من اساليب القمع و محاكم التفتيش و برده تمسكوا بتقاليدهم و مابين السطور كانك عايش. معاهم. واخيرا اجبار اهل البلد علي الرحيل. نهايات كل ابطال الرواية نهايات ماساوية لكنها واقعية في الفترة دي. اتمني اني اشوفها في عمل سينماءي رغم انه صعب بس بجد الفترة دي مظلومة تاريخيا'

In [9]:
# Convert text labels to integers
def convert_labels(df):
    # Handle different possible label formats
    label_mapping = {
        'positive': 0, 'Positive': 0, 'POSITIVE': 0,
        'mixed': 1, 'Mixed': 1, 'MIXED': 1,
        'negative': 2, 'Negative': 2, 'NEGATIVE': 2
    }
    df['label'] = df['label'].map(label_mapping)
    return df

train_df = convert_labels(train_df)
val_df = convert_labels(val_df)
test_df = convert_labels(test_df)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
print(f"Label distribution in train:\n{train_df['label'].value_counts().sort_index()}\n")


Train: 79999 | Validation: 10000 | Test: 10000
Label distribution in train:
label
0    26557
1    26775
2    26667
Name: count, dtype: int64



In [10]:
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df)
})


In [11]:
# ============================================================================
# Tokenization
# ============================================================================
print("Tokenizing datasets...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=192)

tokenized_data = dataset.map(preprocess_function, batched=True)
dataCollator = DataCollatorWithPadding(tokenizer=tokenizer)


Tokenizing datasets...


tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/79999 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [12]:
# ============================================================================
# Model Setup
# ============================================================================
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

Loading model...


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value"]
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 444,675 || all params: 135,640,326 || trainable%: 0.3278


In [14]:
# ============================================================================
# Evaluation Metrics
# ============================================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted")
    }

In [15]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",      # save every N steps
    save_steps=500,              # adjust based on batch size & dataset
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    report_to="none",
    seed=42,
    gradient_accumulation_steps=8,
    fp16=True
)


In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['validation'],
    data_collator=dataCollator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

/tmp/ipython-input-3530230751.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [17]:
# ============================================================================
# Training
# ============================================================================
print("Starting training...\n")
trainer.train(resume_from_checkpoint=True)

Starting training...



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
2,0.608500,0.613838,0.726400,0.725418,0.725751
3,0.613700,0.598684,0.730300,0.728268,0.728605
4,0.564400,0.599753,0.733900,0.732701,0.733032


TrainOutput(global_step=20000, training_loss=0.4398921502113342, metrics={'train_runtime': 6833.4412, 'train_samples_per_second': 46.828, 'train_steps_per_second': 2.927, 'total_flos': 1.47614664483216e+16, 'train_loss': 0.4398921502113342, 'epoch': 4.0})

In [31]:
# ============================================================================
# Evaluation on Test Set
# ============================================================================
print("\n" + "="*60)
print("Test Set Evaluation")
print("="*60)

test_results = trainer.evaluate(tokenized_data['test'])
print(f"\nAccuracy: {test_results['eval_accuracy']:.4f}")
print(f"F1 Macro: {test_results['eval_f1_macro']:.4f}")
print(f"F1 Weighted: {test_results['eval_f1_weighted']:.4f}")

# Detailed classification report
predictions = trainer.predict(tokenized_data['test'])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\n" + classification_report(
    true_labels,
    pred_labels,
    target_names=list(id2label.values()),
    digits=4
))



Test Set Evaluation



Accuracy: 0.7452
F1 Macro: 0.7430
F1 Weighted: 0.7443

              precision    recall  f1-score   support

    Positive     0.7661    0.8226    0.7934      3428
       Mixed     0.6538    0.6304    0.6419      3247
    Negative     0.8109    0.7774    0.7938      3325

    accuracy                         0.7452     10000
   macro avg     0.7436    0.7435    0.7430     10000
weighted avg     0.7445    0.7452    0.7443     10000



In [22]:

# ============================================================================
# Save Model
# ============================================================================
print("="*60)
print("Saving final model and tokenizer...")
trainer.save_model("/content/drive/MyDrive/araBERT")
tokenizer.save_pretrained("/content/drive/MyDrive/tokenizer")
print(f"✓ Model saved to: drive ")
print("="*60)

Saving final model and tokenizer...
✓ Model saved to: tokemizer


In [23]:
print("Loading model...")
model2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading model...


In [24]:
# Freeze base model parameters, unfreeze pooler layer
for param in model2.base_model.parameters():
    param.requires_grad = False

for name, param in model2.base_model.named_parameters():
    if 'pooler' in name:
        param.requires_grad = True

In [25]:
# Display trainable parameters
trainable_params = sum(p.numel() for p in model2.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model2.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)\n")


Trainable: 592,899 / 135,195,651 (0.44%)



In [26]:
trainer2 = Trainer(
    model=model2,
    args=training_args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['validation'],
    data_collator=dataCollator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

/tmp/ipython-input-3061797543.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer2 = Trainer(


In [27]:
# ============================================================================
# Training
# ============================================================================
print("Starting training...\n")
trainer2.train()


Starting training...



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.812500,0.804542,0.627200,0.632593,0.632840
2,0.807000,0.767664,0.651000,0.644438,0.644878
3,0.799400,0.754864,0.659000,0.653885,0.654295
4,0.728800,0.750788,0.660500,0.657245,0.657635


TrainOutput(global_step=20000, training_loss=0.815340189743042, metrics={'train_runtime': 3099.8299, 'train_samples_per_second': 103.23, 'train_steps_per_second': 6.452, 'total_flos': 1.46852231843016e+16, 'train_loss': 0.815340189743042, 'epoch': 4.0})

In [29]:
# Detailed classification report
predictions = trainer2.predict(tokenized_data['test'])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\n" + classification_report(
    true_labels,
    pred_labels,
    target_names=list(id2label.values()),
    digits=4
))


              precision    recall  f1-score   support

    Positive     0.6822    0.7652    0.7213      3428
       Mixed     0.5787    0.5094    0.5419      3247
    Negative     0.7404    0.7341    0.7372      3325

    accuracy                         0.6718     10000
   macro avg     0.6671    0.6696    0.6668     10000
weighted avg     0.6679    0.6718    0.6683     10000



In [30]:
# ============================================================================
# Save Model
# ============================================================================
print("="*60)
print("Saving final model and tokenizer...")
trainer.save_model("/content/drive/MyDrive/araBERT_without_LoRA")
print(f"✓ Model saved to: drive")
print("="*60)

Saving final model and tokenizer...
✓ Model saved to: tokemizer
